In [47]:
import kagglehub
import pandas as pd
import os
import ast

Ucitavanje sirovih podataka

In [64]:
path = kagglehub.dataset_download("wafaaelhusseini/extended-recipes-dataset-64k-dishes")
csv_path = os.path.join(path, "recipes_extended.csv")
first_df = pd.read_csv(csv_path)

path = kagglehub.dataset_download("sooryaprakash12/cleaned-indian-recipes-dataset")
csv_path = os.path.join(path, 'Cleaned_Indian_Food_Dataset.csv')
second_df = pd.read_csv(csv_path)

target_cuisines = ['italian']

def has_target_cuisine(cuisine_list_str):
    try:
        cuisines = ast.literal_eval(cuisine_list_str) if isinstance(cuisine_list_str, str) else []
    except (ValueError, SyntaxError):
        cuisines = []
    return any(c in cuisines[:3] for c in target_cuisines)

first_df = first_df[first_df['cuisine_list'].apply(has_target_cuisine)]
first_df['cuisine'] = first_df['cuisine_list'].apply(lambda x: 'Italian' if 'italian' in ast.literal_eval(x) else 'Indian')
first_df_filtered = first_df[['recipe_title', 'ingredients', 'directions_text', 'cuisine']].copy()
first_df_filtered['ingredients'] = first_df_filtered['ingredients'].apply(lambda x: ', '.join(ast.literal_eval(x)) if isinstance(x, str) else x)

indian_cuisines = [
    'Indian', 'North Indian Recipes', 'South Indian Recipes'
]
italian_cuisines = ['Italian Recipes']
second_df = second_df[second_df['Cuisine'].isin(indian_cuisines + italian_cuisines)]
second_df['Cuisine'] = second_df['Cuisine'].apply(lambda x: 'Italian' if x in italian_cuisines else 'Indian')
second_df_filtered = second_df[['TranslatedRecipeName', 'TranslatedIngredients', 'TranslatedInstructions', 'Cuisine']]

first_df_renamed = first_df_filtered.rename(columns={
    'recipe_title': 'RecipeName',
    'ingredients': 'Ingredients',
    'directions_text': 'Instructions',
    'cuisine': 'Cuisine'
})

second_df_renamed = second_df_filtered.rename(columns={
    'TranslatedRecipeName': 'RecipeName',
    'TranslatedIngredients': 'Ingredients',
    'TranslatedInstructions': 'Instructions',
    'Cuisine': 'Cuisine'
})

combined_df = pd.concat([first_df_renamed, second_df_renamed], ignore_index=True)
combined_df.to_csv('data/recipes_raw.csv', index=False)
combined_df['Cuisine'].value_counts()

Cuisine
Italian    2340
Indian     2248
Name: count, dtype: int64